In [1]:
import sys
import pandas as pd
import sklearn

print("Python environment:", sys.executable)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python environment: /Users/tochi/Brian_ML_Project/.venv/bin/python
pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
#Imports Path to build a path to the Excel file.
from pathlib import Path

#Locates the dataset containing unemployment rates 
file_path = Path("data") / "Eurostat_Data_set.xlsx"

# Read the sheet containing unemployment rates for both sexes combined.
# Keep all rows as data because the first rows contain descriptive information.
raw_data = pd.read_excel(file_path, sheet_name="Sheet 1", header=None)

# Display rows around the year headings and the first country entries.
# iloc uses positions starting at zero; the ending positions are excluded.
# This selects Excel rows 11–17 and the first 9 columns.
display(raw_data.iloc[10:17, :9])

/Users/tochi/Brian_ML_Project/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8
10,TIME,1983,NaN,1984,NaN,1985,NaN,1986,NaN
11,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,European Union - 27 countries (from 2020),:,NaN,:,NaN,:,NaN,:,NaN
13,Euro area – 21 countries (from 2026),:,NaN,:,NaN,:,NaN,:,NaN
14,Belgium,:,NaN,:,NaN,:,NaN,:,NaN
15,Bulgaria,:,NaN,:,NaN,:,NaN,:,NaN
16,Czechia,:,NaN,:,NaN,:,NaN,:,NaN


In [3]:
# Read the row containing the years.
# Position 10 corresponds to Excel row 11.
year_row = raw_data.iloc[10]

# Convert year labels to numbers.
# Labels such as "TIME" and empty cells become NaN.
year_numbers = pd.to_numeric(year_row, errors="coerce")

# Find the columns containing years within our project period.
selected_columns = year_numbers[
    year_numbers.between(2005, 2025)
].index.tolist()

# Confirm which years we selected.
selected_years = year_numbers[selected_columns].astype(int).tolist()
print("Selected years:", selected_years)
print("Number of years:", len(selected_years))

# Preview the country column and selected unemployment-rate columns.
# Position 12 is where the country entries begin.
preview = raw_data.iloc[12:18][[0] + selected_columns].copy()

# Give the preview readable column names.
preview.columns = ["Country"] + selected_years


# You may need to scroll horizontally to see the whole table.
with pd.option_context("display.max_columns", None):
    display(preview)

Selected years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of years: 21


,Country,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
12,European Union - 27 countries (from 2020),5,4.5,3.9,3.7,4.7,5.2,5.3,6,6.4,6.3,5.8,5.2,4.6,4.2,4,4.4,4.2,3.5,3.5,3.5,3.6
13,Euro area – 21 countries (from 2026),5.1,4.6,4.1,3.9,5,5.5,5.6,6.3,6.9,6.8,6.3,5.7,5.1,4.7,4.4,4.9,4.7,4,3.9,3.9,4
14,Belgium,3.7,3.7,3.3,3.2,3.8,4,3.4,3.4,4.3,4.2,4.1,3.7,3.8,3.2,2.9,3.2,3.2,2.8,2.7,2.9,3.5
15,Bulgaria,4.1,3.8,2.2,2.1,2.9,4.4,4.8,5.6,6.1,5,3.7,3.3,2.9,2.2,1.8,2.4,1.8,1.6,1.8,1.6,1.4
16,Czechia,2,2.2,1.5,1.5,2.2,2.5,2.6,2.6,2.5,2.6,2.2,1.8,1.4,1.1,1,1.4,1.4,0.9,1.3,1.3,1.3
17,Denmark,3.7,3.2,3,2.2,3.8,4.6,5,4.8,4.7,4.6,4.6,4.5,4.4,4,4.1,4.6,3.9,3.4,3.8,4.6,5.1


In [4]:
# List the EU countries included in the proposal.
# Malta is excluded because of missing observations in the selected period.
selected_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus",
    "Czechia", "Denmark", "Estonia", "Finland", "France",
    "Germany", "Greece", "Hungary", "Ireland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Netherlands",
    "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden"
]

# Identify rows matching our country list.
# This excludes aggregate rows, other countries and explanatory notes.
country_mask = raw_data[0].isin(selected_countries)

# Extract country names and the rate columns for 2005–2025.
rates = raw_data.loc[
    country_mask, [0] + selected_columns
].copy()

# Replace the original column numbers with readable labels.
rates.columns = ["Country"] + selected_years

# Each rate column in this spreadsheet has a flag column immediately after it.
flag_columns = [column + 1 for column in selected_columns]

# Preserve the flags separately, using the same countries and years.
flags = raw_data.loc[
    country_mask, [0] + flag_columns
].copy()

flags.columns = ["Country"] + selected_years

# Sort both tables alphabetically and give them consecutive row numbers.
rates = rates.sort_values("Country").reset_index(drop=True)
flags = flags.sort_values("Country").reset_index(drop=True)

# Check that every selected country was found exactly once.
assert len(rates) == 26, "Expected 26 country rows."
assert rates["Country"].nunique() == 26, "Check for duplicate countries."

# Confirm the table size and preview the first five countries.
print("Countries selected:", len(rates))
print("Year columns:", len(selected_years))
print("Country-year observations:", len(rates) * len(selected_years))


# Display all 26 countries and all selected years, including 2014 and 2015.
# These settings prevent pandas from hiding rows or columns behind "...".
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(rates)

Countries selected: 26
Year columns: 21
Country-year observations: 546


,Country,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Austria,2.9,2.8,2.6,1.9,2.5,2.4,2.5,2.3,3.4,3.7,3.6,3.4,3,3,2.8,3.2,3.8,3,3.1,3.4,3.6
1,Belgium,3.7,3.7,3.3,3.2,3.8,4,3.4,3.4,4.3,4.2,4.1,3.7,3.8,3.2,2.9,3.2,3.2,2.8,2.7,2.9,3.5
2,Bulgaria,4.1,3.8,2.2,2.1,2.9,4.4,4.8,5.6,6.1,5,3.7,3.3,2.9,2.2,1.8,2.4,1.8,1.6,1.8,1.6,1.4
3,Croatia,5.6,5.1,5.6,4.3,5.3,7.8,8.7,9.9,10,9.1,8.8,7.2,6.5,5.7,4.9,4.6,4.3,4.6,3.5,2.9,2.5
4,Cyprus,3.6,3.5,2.8,2.4,3.5,4.6,5.8,8.7,10.9,10.8,10.7,9.6,8.8,6.9,5.7,6.4,5.8,5.1,5,3.9,3.4
5,Czechia,2,2.2,1.5,1.5,2.2,2.5,2.6,2.6,2.5,2.6,2.2,1.8,1.4,1.1,1,1.4,1.4,0.9,1.3,1.3,1.3
6,Denmark,3.7,3.2,3,2.2,3.8,4.6,5,4.8,4.7,4.6,4.6,4.5,4.4,4,4.1,4.6,3.9,3.4,3.8,4.6,5.1
7,Estonia,4,3.1,2.4,2.8,6.2,9.2,7.9,5.9,5.5,4.7,3.9,3.7,3.3,3.5,2.6,4.7,3.5,3.7,3.6,4.9,4.8
8,Finland,4.4,3.7,3.6,3.3,4,4.4,4,3.9,4.5,5.1,6,5.8,5.3,4.2,3.9,4.3,4.6,4,3.9,4.5,4.9
9,France,5.1,4.8,4.7,3.8,4.8,4.7,4.7,4.8,5.3,5.8,5.7,5.1,4.8,5,4.7,4.7,4.7,4.3,4.5,4.4,4.7


In [5]:
# Convert the selected rate columns into numbers.
# Missing-value markers such as ":" become NaN.
# Country names and the separate flags table remain unchanged.
rates[selected_years] = rates[selected_years].apply(
    pd.to_numeric,
    errors="coerce"
)

# Count missing unemployment rates for each country.
missing_by_country = (
    rates.set_index("Country")[selected_years]
    .isna()
    .sum(axis=1)
)

# Count missing rates across the whole selected dataset.
total_missing = missing_by_country.sum()
print("Total missing unemployment rates:", total_missing)

# Show affected countries, or confirm that no rates are missing.
if total_missing > 0:
    print("Countries with missing rates:")
    display(missing_by_country[missing_by_country > 0])
else:
    print("All 26 countries have a numeric rate for every selected year.")

# Check for values outside the possible percentage range.
outside_range = (
    (rates[selected_years] < 0)
    | (rates[selected_years] > 100)
)

print("Rates outside 0–100%:", outside_range.sum().sum())

Total missing unemployment rates: 0
All 26 countries have a numeric rate for every selected year.
Rates outside 0–100%: 0


In [6]:
# Reshape the flags table so each row represents one country and year.
flags_long = flags.melt(
    id_vars="Country",
    value_vars=selected_years,
    var_name="Year",
    value_name="Flag"
)

# Standardise flag text while preserving missing values.
flags_long["Flag"] = flags_long["Flag"].astype("string").str.strip()

# Select observations with a non-empty flag.
has_flag = flags_long["Flag"].notna() & flags_long["Flag"].ne("")
flagged_observations = flags_long.loc[has_flag].copy()

# Count how many observations have each flag or flag combination.
print("Number of flagged observations:", len(flagged_observations))
display(flagged_observations["Flag"].value_counts())

# Preview the countries and years associated with these flags.
display(flagged_observations.head(15))

# Display the explanatory notes at the bottom of the original sheet.
# These should help us interpret the flag codes.
display(raw_data.iloc[50:, :5])

Number of flagged observations: 139


Flag
b     123
d       7
bu      3
u       3
bd      3
Name: count, dtype: Int64

,Country,Year,Flag
0,Austria,2005,b
1,Belgium,2005,b
2,Bulgaria,2005,b
3,Croatia,2005,bu
4,Cyprus,2005,b
5,Czechia,2005,b
6,Denmark,2005,b
7,Estonia,2005,b
8,Finland,2005,b
9,France,2005,b


,0,1,2,3,4
50,NaN,NaN,NaN,NaN,NaN
51,Special value,NaN,NaN,NaN,NaN
52,:,not available,NaN,NaN,NaN
53,Observation flags:,NaN,NaN,NaN,NaN
54,bd,"break in time series, definition differs (see ...",NaN,NaN,NaN
55,bu,"break in time series, low reliability",NaN,NaN,NaN
56,b,break in time series,NaN,NaN,NaN
57,d,definition differs (see metadata),NaN,NaN,NaN
58,u,low reliability,NaN,NaN,NaN


In [7]:
# Count flagged observations by year and flag combination.
# This helps us identify years where flags affect several countries.
flags_by_year = pd.crosstab(
    flagged_observations["Year"],
    flagged_observations["Flag"]
)

display(flags_by_year)

# Select flags containing "u" or "d", including combinations such as "bu".
# These indicate low reliability or a differing definition.
needs_review = flagged_observations[
    flagged_observations["Flag"].str.contains("u|d", na=False)
].sort_values(["Country", "Year"])

# Display all affected country-year observations.
with pd.option_context("display.max_rows", None):
    display(needs_review)

Flag,b,bd,bu,d,u
Year,,,,,
2005,24,0,2,0,0
2006,7,0,1,0,1
2007,6,0,0,0,1
2008,4,0,0,0,1
2009,3,0,0,0,0
2010,5,0,0,0,0
2011,6,0,0,0,0
2012,1,0,0,0,0
2013,2,0,0,0,0


,Country,Year,Flag
3,Croatia,2005,bu
29,Croatia,2006,bu
425,France,2021,bd
451,France,2022,d
477,France,2023,d
503,France,2024,bd
529,France,2025,d
16,Lithuania,2005,bu
42,Lithuania,2006,u
68,Lithuania,2007,u


In [8]:
# Reshape unemployment rates into one row per country and year.
rates_long = rates.melt(
    id_vars="Country",
    value_vars=selected_years,
    var_name="Year",
    value_name="UnemploymentRate"
)

# Attach each observation's quality flag using country and year.
# Validate that each country-year appears only once in each table.
data = rates_long.merge(
    flags_long,
    on=["Country", "Year"],
    how="left",
    validate="one_to_one"
)

# Store years as integers and put observations in chronological order.
data["Year"] = data["Year"].astype(int)
data = data.sort_values(["Country", "Year"]).reset_index(drop=True)

# Confirm the number of observations and inspect the resulting structure.
print("Country-year observations:", len(data))

#Checking the first 10 observations for verification
# ß

Country-year observations: 546


In [9]:
# Reshape unemployment rates into one row per country and year.
rates_long = rates.melt(
    id_vars="Country",
    value_vars=selected_years,
    var_name="Year",
    value_name="UnemploymentRate"
)

# Attach each observation's quality flag using country and year.
# Validate that each country-year appears only once in each table.
data = rates_long.merge(
    flags_long,
    on=["Country", "Year"],
    how="left",
    validate="one_to_one"
)

# Store years as integers and put observations in chronological order.
data["Year"] = data["Year"].astype(int)
data = data.sort_values(["Country", "Year"]).reset_index(drop=True)

# Confirm the number of observations and inspect the resulting structure.
print("Country-year observations:", len(data))
#Checking the first 10 observations for verification
# display(data.head(10))

Country-year observations: 546


In [10]:
# Reshape unemployment rates into one row per country and year.
rates_long = rates.melt(
    id_vars="Country",
    value_vars=selected_years,
    var_name="Year",
    value_name="UnemploymentRate"
)

# Attach each observation's quality flag using country and year.
# Validate that each country-year appears only once in each table.
data = rates_long.merge(
    flags_long,
    on=["Country", "Year"],
    how="left",
    validate="one_to_one"
)

# Store years as integers and put observations in chronological order.
data["Year"] = data["Year"].astype(int)
data = data.sort_values(["Country", "Year"]).reset_index(drop=True)

# Confirm the number of observations and inspect the resulting structure.
print("Country-year observations:", len(data))

#Checking the first 10 observations for verification
# display(data.head(10))

Country-year observations: 546


In [11]:
# Check that each country-year combination appears only once.
assert not data.duplicated(["Country", "Year"]).any(), \
    "Duplicate country-year rows found."

# Check that every country has all 21 expected years.
expected_years = set(range(2005, 2026))

years_complete = data.groupby("Country")["Year"].apply(
    lambda years: set(years) == expected_years
)

assert years_complete.all(), "Some countries have missing or unexpected years."

# Check that every unemployment rate is present and between 0% and 100%.
assert data["UnemploymentRate"].notna().all(), \
    "Missing unemployment rates found."

assert data["UnemploymentRate"].between(0, 100).all(), \
    "An unemployment rate is outside the valid percentage range."

# Confirm that the prepared table matches the planned size.
assert data["Country"].nunique() == 26, "Expected 26 countries."
assert len(data) == 546, "Expected 546 country-year observations."

print("Checks passed: 26 countries, 21 years each, no duplicates or missing rates.")

Checks passed: 26 countries, 21 years each, no duplicates or missing rates.


In [12]:
# Create a separate table for preparing model inputs.
# Sort by country and year so previous rows represent previous years.
model_data = data.copy()
model_data = model_data.sort_values(
    ["Country", "Year"]
).reset_index(drop=True)

# Create historical inputs separately within each country.
# Grouping prevents one country's values from entering another country's inputs.
for lag in [1, 2, 3]:
    model_data[f"Lag{lag}"] = (
        model_data.groupby("Country")["UnemploymentRate"].shift(lag)
    )

    # Preserve the flag attached to each historical input.
    # The existing Flag column describes the prediction target.
    model_data[f"Lag{lag}_Flag"] = (
        model_data.groupby("Country")["Flag"].shift(lag)
    )

# Keep target years with all three historical rates available.
# Both the one-year and three-year experiments will use these same rows.
model_data = model_data.dropna(
    subset=["Lag1", "Lag2", "Lag3"]
).reset_index(drop=True)

# Check the expected number of usable prediction examples.
assert len(model_data) == 468, "Expected 468 prediction examples."

print("Prediction examples:", len(model_data))
print(
    "Prediction years:",
    model_data["Year"].min(),
    "to",
    model_data["Year"].max()
)

# Inspect the first rows to check that historical inputs align correctly.
display(
    model_data[
        ["Country", "Year", "UnemploymentRate", "Lag1", "Lag2", "Lag3"]
    ].head()
)

Prediction examples: 468
Prediction years: 2008 to 2025


,Country,Year,UnemploymentRate,Lag1,Lag2,Lag3
0,Austria,2008,1.9,2.6,2.8,2.9
1,Austria,2009,2.5,1.9,2.6,2.8
2,Austria,2010,2.4,2.5,1.9,2.6
3,Austria,2011,2.5,2.4,2.5,1.9
4,Austria,2012,2.3,2.5,2.4,2.5


In [13]:
# Use earlier prediction years for learning the model parameters.
train_data = model_data.loc[
    model_data["Year"].between(2008, 2019)
].copy()

# Reserve the following years for choosing model settings.
validation_data = model_data.loc[
    model_data["Year"].between(2020, 2022)
].copy()

# Reserve the latest years for final evaluation.
# Do not use test results to choose models or settings.
test_data = model_data.loc[
    model_data["Year"].between(2023, 2025)
].copy()

# Check that the split sizes match the proposal.
assert len(train_data) == 312, "Expected 312 training examples."
assert len(validation_data) == 78, "Expected 78 validation examples."
assert len(test_data) == 78, "Expected 78 test examples."

# Check that the prediction periods are strictly ordered.
assert train_data["Year"].max() < validation_data["Year"].min()
assert validation_data["Year"].max() < test_data["Year"].min()

# Display the number of examples in each split.
print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Test examples:", len(test_data))

Training examples: 312
Validation examples: 78
Test examples: 78


In [14]:
#Baseline Model


# Import functions for measuring prediction errors.
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Get the actual unemployment rates for validation years 2020–2022.
y_validation = validation_data["UnemploymentRate"]

# Predict each rate by copying that country's previous-year rate.
baseline_predictions = validation_data["Lag1"].copy()

# Calculate the average absolute prediction error.
baseline_mae = mean_absolute_error(
    y_validation,
    baseline_predictions
)

# Calculate RMSE by taking the square root of mean squared error.
baseline_rmse = mean_squared_error(
    y_validation,
    baseline_predictions
) ** 0.5

# Report both errors in percentage points.
print(f"Baseline validation MAE: {baseline_mae:.3f} percentage points")
print(f"Baseline validation RMSE: {baseline_rmse:.3f} percentage points")

Baseline validation MAE: 0.559 percentage points
Baseline validation RMSE: 0.711 percentage points


In [15]:
# Use the previous year's unemployment rate as the single input.
# Double brackets keep the inputs as a two-dimensional table,
# which scikit-learn expects.
X_train_1year = train_data[["Lag1"]].copy()
X_validation_1year = validation_data[["Lag1"]].copy()

# The target is the actual unemployment rate for the prediction year.
y_train = train_data["UnemploymentRate"].copy()
y_validation = validation_data["UnemploymentRate"].copy()

# Check the number of rows and input columns.
print("Training input shape:", X_train_1year.shape)
print("Validation input shape:", X_validation_1year.shape)

# Preview the historical input alongside the value we want to predict.
display(
    train_data[
        ["Country", "Year", "Lag1", "UnemploymentRate"]
    ].head()
)

Training input shape: (312, 1)
Validation input shape: (78, 1)


,Country,Year,Lag1,UnemploymentRate
0,Austria,2008,2.6,1.9
1,Austria,2009,1.9,2.5
2,Austria,2010,2.5,2.4
3,Austria,2011,2.4,2.5
4,Austria,2012,2.5,2.3


In [16]:
# Import the tools for scaling inputs and training Ridge regression.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Define a small set of regularisation strengths before comparing results.
# Larger alpha values apply a stronger penalty to the model coefficients.
alpha_values = [0.01, 0.1, 1.0, 10.0, 100.0]

# Store validation results and fitted models for each setting.
ridge_1year_results = []
ridge_1year_models = {}

# Train and evaluate one model for each regularisation strength.
for alpha in alpha_values:
    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    # Learn scaling statistics and model coefficients from training data only.
    ridge_pipeline.fit(X_train_1year, y_train)

    # Predict validation rates using the fitted pipeline.
    predictions = ridge_pipeline.predict(X_validation_1year)

    # Calculate validation errors in percentage points.
    mae = mean_absolute_error(y_validation, predictions)
    rmse = mean_squared_error(y_validation, predictions) ** 0.5

    # Record the scores and retain the fitted pipeline.
    ridge_1year_results.append({
        "Alpha": alpha,
        "Validation_MAE": mae,
        "Validation_RMSE": rmse
    })
    ridge_1year_models[alpha] = ridge_pipeline

# Rank settings by validation MAE.
# If MAE values tie exactly, prefer the smaller alpha.
ridge_1year_results = pd.DataFrame(ridge_1year_results).sort_values(
    ["Validation_MAE", "Alpha"]
).reset_index(drop=True)

# Keep the model with the lowest validation MAE.
best_alpha_1year = float(ridge_1year_results.loc[0, "Alpha"])
best_ridge_1year = ridge_1year_models[best_alpha_1year]

# Display the results and compare the selected model with the baseline.
display(ridge_1year_results)

best_mae_1year = ridge_1year_results.loc[0, "Validation_MAE"]

print("Selected alpha:", best_alpha_1year)
print(f"Ridge validation MAE: {best_mae_1year:.3f} percentage points")
print(f"Baseline validation MAE: {baseline_mae:.3f} percentage points")
print(
    "Ridge improves on baseline MAE:",
    best_mae_1year < baseline_mae
)

,Alpha,Validation_MAE,Validation_RMSE
0,0.01,0.561727,0.694370
1,0.10,0.561729,0.694286
2,1.00,0.561746,0.693480
3,10.00,0.562102,0.688973
4,100.00,0.708348,0.836834


Selected alpha: 0.01
Ridge validation MAE: 0.562 percentage points
Baseline validation MAE: 0.559 percentage points
Ridge improves on baseline MAE: False


In [18]:
# Use the previous three years as separate input columns.
X_train_3year = train_data[["Lag1", "Lag2", "Lag3"]].copy()
X_validation_3year = validation_data[["Lag1", "Lag2", "Lag3"]].copy()

# Confirm that both experiments use exactly the same rows.
assert X_train_3year.index.equals(X_train_1year.index)
assert X_validation_3year.index.equals(X_validation_1year.index)

# Store results and fitted models separately from the one-year experiment.
ridge_3year_results = []
ridge_3year_models = {}

# Reuse the same alpha candidates as the one-year experiment.
for alpha in alpha_values:
    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    # Learn scaling statistics and coefficients from training data only.
    ridge_pipeline.fit(X_train_3year, y_train)

    # Predict validation rates without refitting the scaler or model.
    predictions = ridge_pipeline.predict(X_validation_3year)

    # Measure validation errors in percentage points.
    mae = mean_absolute_error(y_validation, predictions)
    rmse = mean_squared_error(y_validation, predictions) ** 0.5

    # Save the scores and fitted pipeline for this alpha.
    ridge_3year_results.append({
        "Alpha": alpha,
        "Validation_MAE": mae,
        "Validation_RMSE": rmse
    })
    ridge_3year_models[alpha] = ridge_pipeline

# Rank settings by validation MAE.
# If MAE values tie exactly, prefer the smaller alpha.
ridge_3year_results = pd.DataFrame(ridge_3year_results).sort_values(
    ["Validation_MAE", "Alpha"]
).reset_index(drop=True)

# Retain the fitted model with the lowest validation MAE.
best_alpha_3year = float(ridge_3year_results.loc[0, "Alpha"])
best_ridge_3year = ridge_3year_models[best_alpha_3year]

# Display the tuning results and selected regularisation strength.
display(ridge_3year_results)
print("Selected alpha:", best_alpha_3year)

# Compare the baseline with the selected model from each Ridge experiment.
validation_comparison = pd.DataFrame([
    {
        "Model": "Previous-year baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse
    },
    {
        "Model": "Ridge — one-year input",
        "MAE": ridge_1year_results.loc[0, "Validation_MAE"],
        "RMSE": ridge_1year_results.loc[0, "Validation_RMSE"]
    },
    {
        "Model": "Ridge — three-year inputs",
        "MAE": ridge_3year_results.loc[0, "Validation_MAE"],
        "RMSE": ridge_3year_results.loc[0, "Validation_RMSE"]
    }
])

# Identify models with lower validation MAE than the baseline.
# Comparisons use the full, unrounded scores.
validation_comparison["Beats_baseline_MAE"] = (
    validation_comparison["MAE"] < baseline_mae
)

# Separately identify models with lower validation RMSE than the baseline.
validation_comparison["Beats_baseline_RMSE"] = (
    validation_comparison["RMSE"] < baseline_rmse
)

# Show all comparison columns without hiding any behind "...".
with pd.option_context("display.max_columns", None):
    display(validation_comparison)

,Alpha,Validation_MAE,Validation_RMSE
0,100.00,0.587157,0.702025
1,10.00,0.627170,0.759927
2,1.00,0.668413,0.829279
3,0.10,0.677220,0.851674
4,0.01,0.678241,0.854493


Selected alpha: 100.0


,Model,MAE,RMSE,Beats_baseline_MAE,Beats_baseline_RMSE
0,Previous-year baseline,0.558974,0.711265,False,False
1,Ridge — one-year input,0.561727,0.694370,False,True
2,Ridge — three-year inputs,0.587157,0.702025,False,True


In [19]:
# Import the neural network, timing and warning tools.
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from time import perf_counter
import warnings

# Define a small search before looking at the results.
# Each model has one hidden layer containing either 4 or 8 neurons.
hidden_sizes = [4, 8]
mlp_alphas = [0.001, 0.1, 1.0]
random_seeds = [0, 1, 2, 3, 4]

# Store every run's results and fitted pipeline.
mlp_1year_runs = []
mlp_1year_models = {}

for neurons in hidden_sizes:
    for alpha in mlp_alphas:
        for seed in random_seeds:

            # Scale inputs and train a small neural network.
            # Keep our existing chronological validation set.
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("mlp", MLPRegressor(
                    hidden_layer_sizes=(neurons,),
                    activation="relu",
                    solver="lbfgs",
                    alpha=alpha,
                    max_iter=2000,
                    max_fun=25000,
                    early_stopping=False,
                    random_state=seed
                ))
            ])

            # Record warnings so training problems are not overlooked.
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                # Time fitting, including the input scaler.
                start = perf_counter()
                pipeline.fit(X_train_1year, y_train)
                fit_seconds = perf_counter() - start

            # Record whether the optimiser reported a convergence problem.
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            # Predict validation rates using training-only scaling.
            predictions = pipeline.predict(X_validation_1year)

            # Save errors, timing and warning information for this run.
            mlp_1year_runs.append({
                "Neurons": neurons,
                "Alpha": alpha,
                "Seed": seed,
                "MAE": mean_absolute_error(y_validation, predictions),
                "RMSE": mean_squared_error(
                    y_validation, predictions
                ) ** 0.5,
                "Fit_seconds": fit_seconds,
                "Convergence_warning": convergence_warning,
                "Warning_messages": " | ".join(
                    str(w.message) for w in caught
                )
            })

            # Retain each fitted model for later evaluation.
            mlp_1year_models[(neurons, alpha, seed)] = pipeline

# Keep the individual runs for reproducibility and inspection.
mlp_1year_runs = pd.DataFrame(mlp_1year_runs)

# Summarise errors across seeds for each configuration.
# Standard deviations describe variation between runs, not confidence intervals.
mlp_1year_summary = (
    mlp_1year_runs.groupby(["Neurons", "Alpha"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Std_MAE=("MAE", "std"),
        Mean_RMSE=("RMSE", "mean"),
        Std_RMSE=("RMSE", "std"),
        Mean_fit_seconds=("Fit_seconds", "mean"),
        Convergence_warnings=("Convergence_warning", "sum")
    )
    .sort_values(["Mean_MAE", "Neurons", "Alpha"])
    .reset_index(drop=True)
)

# Compare average validation errors with the previous-year baseline.
mlp_1year_summary["Beats_baseline_MAE"] = (
    mlp_1year_summary["Mean_MAE"] < baseline_mae
)
mlp_1year_summary["Beats_baseline_RMSE"] = (
    mlp_1year_summary["Mean_RMSE"] < baseline_rmse
)

# Display every result column.
with pd.option_context("display.max_columns", None):
    display(mlp_1year_summary)

# Show any warning details before accepting a configuration.
warning_runs = mlp_1year_runs[
    mlp_1year_runs["Warning_messages"].ne("")
]

print("Completed runs:", len(mlp_1year_runs))
print("Runs with warnings:", len(warning_runs))

if not warning_runs.empty:
    display(
        warning_runs[
            ["Neurons", "Alpha", "Seed", "Warning_messages"]
        ]
    )

,Neurons,Alpha,Mean_MAE,Std_MAE,Mean_RMSE,Std_RMSE,Mean_fit_seconds,Convergence_warnings,Beats_baseline_MAE,Beats_baseline_RMSE
0,4,0.001,0.561301,0.001829,0.695053,0.001788,0.081698,0,False,True
1,8,0.001,0.561327,0.000791,0.694379,0.001172,0.062083,0,False,True
2,8,0.100,0.561427,0.000943,0.694896,0.000868,0.072231,0,False,True
3,8,1.000,0.561455,0.000300,0.693971,0.000325,0.078009,0,False,True
4,4,0.100,0.562074,0.002865,0.695723,0.002597,0.063196,0,False,True
5,4,1.000,0.564880,0.004887,0.697310,0.005041,0.059052,0,False,True


Completed runs: 30
Runs with warnings: 0


In [20]:
# Read the settings ranked first by mean validation MAE.
best_mlp_1year_neurons = int(mlp_1year_summary.loc[0, "Neurons"])
best_mlp_1year_alpha = float(mlp_1year_summary.loc[0, "Alpha"])

# Keep the fitted model for each seed under the selected configuration.
best_mlp_1year_models = {
    seed: mlp_1year_models[
        (best_mlp_1year_neurons, best_mlp_1year_alpha, seed)
    ]
    for seed in random_seeds
}

# Confirm the selected configuration.
print("Selected neurons:", best_mlp_1year_neurons)
print("Selected alpha:", best_mlp_1year_alpha)
print("Retained models:", len(best_mlp_1year_models))

Selected neurons: 4
Selected alpha: 0.001
Retained models: 5


In [22]:
# Confirm that the three-year inputs use the same training and validation rows.
assert X_train_3year.index.equals(X_train_1year.index)
assert X_validation_3year.index.equals(X_validation_1year.index)

# Store results and fitted models separately from the one-year experiment.
mlp_3year_runs = []
mlp_3year_models = {}

# Reuse the same hidden-layer sizes, regularisation strengths and seeds.
for neurons in hidden_sizes:
    for alpha in mlp_alphas:
        for seed in random_seeds:

            # Build a pipeline with training-only scaling and one hidden layer.
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("mlp", MLPRegressor(
                    hidden_layer_sizes=(neurons,),
                    activation="relu",
                    solver="lbfgs",
                    alpha=alpha,
                    max_iter=2000,
                    max_fun=25000,
                    early_stopping=False,
                    random_state=seed
                ))
            ])

            # Capture warnings and measure fitting time.
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                start = perf_counter()
                pipeline.fit(X_train_3year, y_train)
                fit_seconds = perf_counter() - start

            # Record any convergence warning from training.
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            # Predict validation rates using the three historical inputs.
            predictions = pipeline.predict(X_validation_3year)

            # Record errors, fitting time and warnings for this run.
            mlp_3year_runs.append({
                "Neurons": neurons,
                "Alpha": alpha,
                "Seed": seed,
                "MAE": mean_absolute_error(y_validation, predictions),
                "RMSE": mean_squared_error(
                    y_validation, predictions
                ) ** 0.5,
                "Fit_seconds": fit_seconds,
                "Convergence_warning": convergence_warning,
                "Warning_messages": " | ".join(
                    str(w.message) for w in caught
                )
            })

            # Retain the fitted pipeline for later evaluation.
            mlp_3year_models[(neurons, alpha, seed)] = pipeline

# Keep the individual run results.
mlp_3year_runs = pd.DataFrame(mlp_3year_runs)

# Summarise each configuration across the five seeds.
mlp_3year_summary = (
    mlp_3year_runs.groupby(["Neurons", "Alpha"], as_index=False)
    .agg(
        Mean_MAE=("MAE", "mean"),
        Std_MAE=("MAE", "std"),
        Mean_RMSE=("RMSE", "mean"),
        Std_RMSE=("RMSE", "std"),
        Mean_fit_seconds=("Fit_seconds", "mean"),
        Convergence_warnings=("Convergence_warning", "sum")
    )
    .sort_values(["Mean_MAE", "Neurons", "Alpha"])
    .reset_index(drop=True)
)

# Compare mean validation errors with the baseline using unrounded scores.
mlp_3year_summary["Beats_baseline_MAE"] = (
    mlp_3year_summary["Mean_MAE"] < baseline_mae
)
mlp_3year_summary["Beats_baseline_RMSE"] = (
    mlp_3year_summary["Mean_RMSE"] < baseline_rmse
)

# Display every column in the results table.
with pd.option_context("display.max_columns", None):
    display(mlp_3year_summary)

# Check for training warnings before accepting the selected settings.
warning_runs_3year = mlp_3year_runs[
    mlp_3year_runs["Warning_messages"].ne("")
]

print("Completed runs:", len(mlp_3year_runs))
print("Runs with warnings:", len(warning_runs_3year))

if not warning_runs_3year.empty:
    display(
        warning_runs_3year[
            ["Neurons", "Alpha", "Seed", "Warning_messages"]
        ]
    )

,Neurons,Alpha,Mean_MAE,Std_MAE,Mean_RMSE,Std_RMSE,Mean_fit_seconds,Convergence_warnings,Beats_baseline_MAE,Beats_baseline_RMSE
0,4,1.000,0.679581,0.016863,0.854300,0.017310,0.079015,0,False,False
1,4,0.100,0.695124,0.033165,0.878804,0.037445,0.092419,0,False,False
2,4,0.001,0.697094,0.030923,0.879869,0.039535,0.137467,0,False,False
3,8,1.000,0.699091,0.044355,0.877828,0.046860,0.118364,0,False,False
4,8,0.100,0.714919,0.036280,0.902722,0.046139,0.151113,0,False,False
5,8,0.001,0.741420,0.024721,0.929573,0.027962,0.133601,0,False,False


Completed runs: 30
Runs with warnings: 0


In [23]:
# Read the configuration with the lowest mean validation MAE.
best_mlp_3year_neurons = int(mlp_3year_summary.loc[0, "Neurons"])
best_mlp_3year_alpha = float(mlp_3year_summary.loc[0, "Alpha"])

# Retain all five fitted models for this configuration.
# Do not select only the seed with the lowest error.
best_mlp_3year_models = {
    seed: mlp_3year_models[
        (best_mlp_3year_neurons, best_mlp_3year_alpha, seed)
    ]
    for seed in random_seeds
}

# Confirm the selected settings and number of retained models.
print("Selected neurons:", best_mlp_3year_neurons)
print("Selected alpha:", best_mlp_3year_alpha)
print("Retained models:", len(best_mlp_3year_models))

Selected neurons: 4
Selected alpha: 1.0
Retained models: 5
